In [ ]:
# TinyLLM for Sentiment Analysis - From Scratch

# Install required packages (run once)
!pip install torch torchvision --quiet
!pip install numpy pandas scikit-learn --quiet
!pip install matplotlib seaborn tqdm --quiet

print("✓ Packages installed successfully!")

# Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import math
import re
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'\n🚀 Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
✓ Packages installed successfully!


In [ ]:
class SimpleTokenizer:
    """Custom tokenizer for text processing"""
    
    def __init__(self, vocab_size=10000):
        self.vocab_size = vocab_size
        self.word2idx = {}
        self.idx2word = {}
        
    def build_vocab(self, texts):
        """Build vocabulary from training texts"""
        # Tokenize and count words
        words = []
        for text in texts:
            # Simple tokenization (you can improve this)
            tokens = text.lower().split()
            words.extend(tokens)
        
        # Get most common words
        word_counts = Counter(words)
        
        # Special tokens + most common words
        vocab = ['<PAD>', '<UNK>', '<SOS>', '<EOS>'] + \
                [word for word, _ in word_counts.most_common(self.vocab_size - 4)]
        
        # Create mappings
        self.word2idx = {word: idx for idx, word in enumerate(vocab)}
        self.idx2word = {idx: word for word, idx in self.word2idx.items()}
        
        print(f"✓ Vocabulary built: {len(self.word2idx)} tokens")
        
    def encode(self, text, max_length=128):
        """Convert text to token indices"""
        tokens = text.lower().split()
        indices = [self.word2idx.get(token, self.word2idx['<UNK>']) 
                   for token in tokens]
        
        # Padding or truncation
        if len(indices) < max_length:
            indices += [self.word2idx['<PAD>']] * (max_length - len(indices))
        else:
            indices = indices[:max_length]
        
        return indices
    
    def decode(self, indices):
        """Convert token indices back to text"""
        return ' '.join([self.idx2word.get(idx, '<UNK>') for idx in indices 
                        if idx != self.word2idx['<PAD>']])

# Test tokenizer
print("Testing tokenizer...")
test_tokenizer = SimpleTokenizer(vocab_size=100)
test_tokenizer.build_vocab(["hello world", "this is a test"])
encoded = test_tokenizer.encode("hello world")
print(f"Encoded: {encoded[:10]}")
print("✓ Tokenizer working!")

In [ ]:
class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding"""
    
    def __init__(self, d_model, max_len=512):
        super().__init__()
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * 
                            (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        # x: [batch_size, seq_len, d_model]
        return x + self.pe[:, :x.size(1), :]

# Test positional encoding
print("Testing positional encoding...")
pos_enc = PositionalEncoding(d_model=64, max_len=128)
test_input = torch.randn(2, 10, 64)  # batch=2, seq_len=10, d_model=64
output = pos_enc(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {output.shape}")
print("✓ Positional encoding working!")

In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head self-attention mechanism"""
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Linear projections
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
    def scaled_dot_product_attention(self, Q, K, V, mask=None):
        """Compute scaled dot-product attention"""
        # Q, K, V: [batch_size, num_heads, seq_len, d_k]
        
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        
        attention_weights = torch.softmax(scores, dim=-1)
        output = torch.matmul(attention_weights, V)
        
        return output, attention_weights
    
    def forward(self, x, mask=None):
        batch_size = x.size(0)
        
        # Linear projections and split into heads
        Q = self.W_q(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)
        
        # Attention
        attn_output, attn_weights = self.scaled_dot_product_attention(Q, K, V, mask)
        
        # Concatenate heads
        attn_output = attn_output.transpose(1, 2).contiguous().view(
            batch_size, -1, self.d_model
        )
        
        # Final linear projection
        output = self.W_o(attn_output)
        
        return output, attn_weights

# Test attention
print("Testing multi-head attention...")
mha = MultiHeadAttention(d_model=64, num_heads=4)
test_input = torch.randn(2, 10, 64)
output, weights = mha(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print("✓ Multi-head attention working!")

In [ ]:
class FeedForward(nn.Module):
    """Position-wise feed-forward network"""
    
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()
        
    def forward(self, x):
        return self.linear2(self.dropout(self.activation(self.linear1(x))))

# Test feed-forward
print("Testing feed-forward network...")
ff = FeedForward(d_model=64, d_ff=256)
test_input = torch.randn(2, 10, 64)
output = ff(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {output.shape}")
print("✓ Feed-forward network working!")

In [ ]:
class TransformerBlock(nn.Module):
    """Single transformer encoder block"""
    
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        # Multi-head attention with residual connection
        attn_output, attn_weights = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        
        # Feed-forward with residual connection
        ff_output = self.ff(x)
        x = self.norm2(x + self.dropout(ff_output))
        
        return x, attn_weights

# Test transformer block
print("Testing transformer block...")
block = TransformerBlock(d_model=64, num_heads=4, d_ff=256)
test_input = torch.randn(2, 10, 64)
output, weights = block(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {output.shape}")
print("✓ Transformer block working!")

In [ ]:
class TinyLLM(nn.Module):
    """Complete Tiny Language Model for Sentiment Analysis"""
    
    def __init__(self, vocab_size, d_model=256, num_heads=4, 
                 num_layers=4, d_ff=1024, max_len=128, 
                 num_classes=3, dropout=0.1):
        super().__init__()
        
        self.d_model = d_model
        
        # Embedding layers
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        
        # Transformer blocks
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        
        # Classification head
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(d_model, num_classes)
        
        # Initialize weights
        self._init_weights()
        
    def _init_weights(self):
        """Initialize model weights"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, x, mask=None):
        # x: [batch_size, seq_len]
        
        # Embedding and positional encoding
        x = self.token_embedding(x) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.dropout(x)
        
        # Pass through transformer blocks
        attention_weights = []
        for block in self.transformer_blocks:
            x, attn_weights = block(x, mask)
            attention_weights.append(attn_weights)
        
        # Global average pooling
        x = x.mean(dim=1)
        
        # Classification
        logits = self.classifier(x)
        
        return logits, attention_weights

print("✓ TinyLLM model defined successfully!")

In [ ]:
class SentimentDataset(Dataset):
    """Custom dataset for sentiment analysis"""
    
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        
        tokens = self.tokenizer.encode(text, self.max_length)
        
        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

print("✓ Dataset class defined!")

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-4):
    """Train the model"""
    
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    
    best_val_acc = 0
    history = {
        'train_loss': [], 'train_acc': [], 
        'val_loss': [], 'val_acc': []
    }
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0
        train_correct = 0
        train_total = 0
        
        train_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        for batch in train_bar:
            input_ids = batch['input_ids'].to(device)
            labels = batch['labels'].to(device)
            
            optimizer.zero_grad()
            
            logits, _ = model(input_ids)
            loss = criterion(logits, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(logits, 1)
            train_total += labels.size(0)
            train_correct += (predicted == labels).sum().item()
            
            train_bar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100*train_correct/train_total:.2f}%'
            })
        
        # Validation phase
        model.eval()
        val_loss = 0
        val_correct = 0
        val_total = 0
        
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]')
            for batch in val_bar:
                input_ids = batch['input_ids'].to(device)
                labels = batch['labels'].to(device)
                
                logits, _ = model(input_ids)
                loss = criterion(logits, labels)
                
                val_loss += loss.item()
                _, predicted = torch.max(logits, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()
                
                val_bar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc': f'{100*val_correct/val_total:.2f}%'
                })
        
        # Calculate metrics
        train_acc = 100 * train_correct / train_total
        val_acc = 100 * val_correct / val_total
        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        
        # Store history
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        
        # Print epoch summary
        print(f'\n{"="*60}')
        print(f'Epoch {epoch+1}/{epochs} Summary:')
        print(f'Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.2f}%')
        print(f'Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%')
        print(f'{"="*60}\n')
        
        scheduler.step()
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), 'best_model.pt')
            print(f'✓ Saved best model (Val Acc: {val_acc:.2f}%)\n')
    
    print(f'\n🎉 Training completed! Best Val Acc: {best_val_acc:.2f}%')
    return history

print("✓ Training function defined!")

In [ ]:
def plot_training_history(history):
    """Plot training and validation metrics"""
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss plot
    ax1.plot(epochs, history['train_loss'], 'b-o', label='Train Loss', linewidth=2)
    ax1.plot(epochs, history['val_loss'], 'r-o', label='Val Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # Accuracy plot
    ax2.plot(epochs, history['train_acc'], 'b-o', label='Train Acc', linewidth=2)
    ax2.plot(epochs, history['val_acc'], 'r-o', label='Val Acc', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

def plot_confusion_matrix(y_true, y_pred, class_names=['Negative', 'Neutral', 'Positive']):
    """Plot confusion matrix"""
    
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.ylabel('True Label', fontsize=12)
    plt.xlabel('Predicted Label', fontsize=12)
    plt.tight_layout()
    plt.show()

def visualize_attention(model, text, tokenizer, layer_idx=0, head_idx=0):
    """Visualize attention weights"""
    
    model.eval()
    tokens = tokenizer.encode(text, max_length=128)
    input_ids = torch.tensor([tokens]).to(device)
    
    with torch.no_grad():
        logits, attention_weights = model(input_ids)
    
    # Get attention from specific layer and head
    attn = attention_weights[layer_idx][0, head_idx].cpu().numpy()
    
    # Get actual tokens (remove padding)
    actual_tokens = [tokenizer.idx2word.get(idx, '<UNK>') 
                     for idx in tokens if idx != tokenizer.word2idx['<PAD>']]
    
    # Limit to actual sequence length
    seq_len = len(actual_tokens)
    attn = attn[:seq_len, :seq_len]
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(attn, cmap='viridis', xticklabels=actual_tokens, 
                yticklabels=actual_tokens, cbar_kws={'label': 'Attention Weight'})
    plt.title(f'Attention Weights - Layer {layer_idx}, Head {head_idx}', 
              fontsize=14, fontweight='bold')
    plt.xlabel('Key', fontsize=12)
    plt.ylabel('Query', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

print("✓ Visualization functions defined!")

In [ ]:
# Sample dataset (replace with your actual dataset)
# For demonstration, using simple examples

sample_data = {
    'positive': [
        "This movie was absolutely fantastic! I loved every minute of it.",
        "Amazing performance by the actors. Highly recommended!",
        "Best film I've seen this year. Truly outstanding.",
        "Wonderful story with great cinematography.",
        "Excellent movie! Will watch again.",
        "Brilliant direction and superb acting.",
        "Loved it! A masterpiece of cinema.",
        "Outstanding film with powerful message.",
        "Incredible experience. Must watch!",
        "Perfect movie for the whole family."
    ] * 10,  # Repeat for more samples
    
    'negative': [
        "Terrible movie. Complete waste of time and money.",
        "Horrible acting and poor storyline.",
        "Worst film I've ever seen. Very disappointing.",
        "Boring and predictable. Don't bother watching.",
        "Awful movie with bad direction.",
        "Completely unwatchable. Save your money.",
        "Disappointing film with terrible script.",
        "Poor execution and weak performances.",
        "Not worth watching at all.",
        "Dreadful movie. Avoid at all costs."
    ] * 10,
    
    'neutral': [
        "It was okay. Nothing special but not terrible either.",
        "Average movie with some good moments.",
        "Decent film but could have been better.",
        "Not bad, but not great either.",
        "Mediocre movie with mixed reviews.",
        "Acceptable entertainment for a lazy evening.",
        "Fair movie with average performances.",
        "Okay storyline but lacks depth.",
        "Moderate film with some interesting parts.",
        "Passable movie, nothing memorable."
    ] * 10
}

# Create dataset
texts = sample_data['positive'] + sample_data['negative'] + sample_data['neutral']
labels = [2] * len(sample_data['positive']) + \
         [0] * len(sample_data['negative']) + \
         [1] * len(sample_data['neutral'])

print(f"📊 Dataset Statistics:")
print(f"Total samples: {len(texts)}")
print(f"Positive: {labels.count(2)}")
print(f"Negative: {labels.count(0)}")
print(f"Neutral: {labels.count(1)}")

# Optional: Load from CSV
# Uncomment if you have a CSV file
"""
df = pd.read_csv('your_dataset.csv')
texts = df['text'].tolist()
labels = df['label'].tolist()  # 0: negative, 1: neutral, 2: positive
"""

In [ ]:
print("🔧 Preparing data...")

# Split data
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

print(f"\n✓ Data split:")
print(f"  Training samples: {len(train_texts)}")
print(f"  Validation samples: {len(val_texts)}")

# Build tokenizer
print("\n🔤 Building vocabulary...")
tokenizer = SimpleTokenizer(vocab_size=5000)
tokenizer.build_vocab(train_texts)

import json

# You must save the mapping so your API knows how to encode raw text
with open('tokenizer_vocab.json', 'w') as f:
    json.dump(tokenizer.word2idx, f)
print("✓ Tokenizer vocabulary saved!")

# Create datasets
train_dataset = SentimentDataset(train_texts, train_labels, tokenizer)
val_dataset = SentimentDataset(val_texts, val_labels, tokenizer)

# Create dataloaders
BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"\n✓ DataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Batch size: {BATCH_SIZE}")

In [ ]:
print("🤖 Initializing TinyLLM model...\n")

# Model configuration
MODEL_CONFIG = {
    'vocab_size': len(tokenizer.word2idx),
    'd_model': 128,        # Embedding dimension
    'num_heads': 4,        # Number of attention heads
    'num_layers': 3,       # Number of transformer layers
    'd_ff': 512,           # Feed-forward dimension
    'max_len': 128,        # Maximum sequence length
    'num_classes': 3,      # Negative, Neutral, Positive
    'dropout': 0.1
}

# Create model
model = TinyLLM(**MODEL_CONFIG)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("📊 Model Statistics:")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Model size: ~{total_params * 4 / (1024**2):.2f} MB")
print(f"\n  Configuration:")
for key, value in MODEL_CONFIG.items():
    print(f"    {key}: {value}")

# Move to device
model = model.to(device)
print(f"\n✓ Model moved to {device}")

In [ ]:
print("🚀 Starting training...\n")

# Training configuration
EPOCHS = 10
LEARNING_RATE = 1e-4

# Train the model
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=EPOCHS,
    lr=LEARNING_RATE
)

print("\n✅ Training completed!")

In [ ]:
# Plot training history
plot_training_history(history)

# Print best results
best_epoch = np.argmax(history['val_acc'])
print(f"\n🏆 Best Results:")
print(f"  Epoch: {best_epoch + 1}")
print(f"  Train Acc: {history['train_acc'][best_epoch]:.2f}%")
print(f"  Val Acc: {history['val_acc'][best_epoch]:.2f}%")
print(f"  Val Loss: {history['val_loss'][best_epoch]:.4f}")

In [ ]:
print("📊 Evaluating model on validation set...\n")

# Load best model
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

# Get predictions
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(val_loader, desc='Evaluating'):
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        
        logits, _ = model(input_ids)
        _, predicted = torch.max(logits, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Classification report
class_names = ['Negative', 'Neutral', 'Positive']
print("\n" + "="*60)
print("Classification Report:")
print("="*60)
print(classification_report(all_labels, all_preds, target_names=class_names))

# Confusion matrix
plot_confusion_matrix(all_labels, all_preds, class_names)

In [ ]:
def predict_sentiment(text, model, tokenizer):
    """Predict sentiment for a single text"""
    model.eval()
    tokens = tokenizer.encode(text, max_length=128)
    input_ids = torch.tensor([tokens]).to(device)
    
    with torch.no_grad():
        logits, _ = model(input_ids)
        probs = torch.softmax(logits, dim=1)
        predicted = torch.argmax(probs, dim=1)
    
    sentiment_map = {0: 'Negative 😞', 1: 'Neutral 😐', 2: 'Positive 😊'}
    
    print(f"\n{'='*60}")
    print(f"Text: {text}")
    print(f"{'='*60}")
    print(f"Predicted Sentiment: {sentiment_map[predicted.item()]}")
    print(f"Confidence: {probs[0][predicted].item()*100:.2f}%")
    print(f"\nProbability Distribution:")
    print(f"  Negative: {probs[0][0].item()*100:.2f}%")
    print(f"  Neutral:  {probs[0][1].item()*100:.2f}%")
    print(f"  Positive: {probs[0][2].item()*100:.2f}%")
    print(f"{'='*60}\n")
    
    return predicted.item(), probs[0].cpu().numpy()

# Test examples
test_examples = [
    "This is the best movie I've ever seen! Absolutely amazing!",
    "Terrible film, complete waste of time. Very disappointed.",
    "It was okay, nothing special but watchable.",
    "Fantastic performance by all actors. Highly recommend!",
    "Boring and predictable. Would not watch again.",
]

print("🧪 Testing model predictions:\n")
for text in test_examples:
    predict_sentiment(text, model, tokenizer)

In [ ]:
# Visualize attention for an example
example_text = "This movie was absolutely fantastic and amazing!"

print(f"Visualizing attention for: '{example_text}'\n")

# Visualize different layers and heads
for layer in range(min(2, MODEL_CONFIG['num_layers'])):
    for head in range(min(2, MODEL_CONFIG['num_heads'])):
        print(f"\nLayer {layer}, Head {head}:")
        visualize_attention(model, example_text, tokenizer, layer, head)

In [ ]:
# Save everything needed for inference
save_dict = {
    'model_state_dict': model.state_dict(),
    'model_config': MODEL_CONFIG,
    'tokenizer_word2idx': tokenizer.word2idx,
    'tokenizer_idx2word': tokenizer.idx2word,
    'training_history': history,
    'class_names': ['Negative', 'Neutral', 'Positive']
}

torch.save(save_dict, 'tinyllm_complete.pt')
print("✅ Complete model saved as 'tinyllm_complete.pt'")

# Save model architecture summary
with open('model_summary.txt', 'w') as f:
    f.write("TinyLLM Model Summary\n")
    f.write("="*60 + "\n\n")
    f.write(f"Total Parameters: {total_params:,}\n")
    f.write(f"Model Size: ~{total_params * 4 / (1024**2):.2f} MB\n\n")
    f.write("Configuration:\n")
    for key, value in MODEL_CONFIG.items():
        f.write(f"  {key}: {value}\n")
    f.write(f"\nBest Validation Accuracy: {max(history['val_acc']):.2f}%\n")

print("✅ Model summary saved as 'model_summary.txt'")

In [ ]:
# Example: How to load and use the saved model later

def load_model(filepath='tinyllm_complete.pt'):
    """Load saved model for inference"""
    
    checkpoint = torch.load(filepath, map_location=device)
    
    # Recreate tokenizer
    tokenizer = SimpleTokenizer()
    tokenizer.word2idx = checkpoint['tokenizer_word2idx']
    tokenizer.idx2word = checkpoint['tokenizer_idx2word']
    
    # Recreate model
    model = TinyLLM(**checkpoint['model_config'])
    model.load_state_dict(checkpoint['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    return model, tokenizer, checkpoint

# Load model
print("Loading saved model...")
loaded_model, loaded_tokenizer, checkpoint = load_model()
print("✅ Model loaded successfully!")

# Test with loaded model
test_text = "This is bad product! Highly not recommended!"
predict_sentiment(test_text, loaded_model, loaded_tokenizer)

In [ ]:
# Analyze model performance

print("📈 Model Performance Analysis\n")
print("="*60)

# Training summary
print("\n1. Training Summary:")
print(f"   Total Epochs: {len(history['train_loss'])}")
print(f"   Final Train Accuracy: {history['train_acc'][-1]:.2f}%")
print(f"   Final Val Accuracy: {history['val_acc'][-1]:.2f}%")
print(f"   Best Val Accuracy: {max(history['val_acc']):.2f}%")
print(f"   Best Epoch: {np.argmax(history['val_acc']) + 1}")

# Model size
print("\n2. Model Size:")
print(f"   Parameters: {total_params:,}")
print(f"   Size: ~{total_params * 4 / (1024**2):.2f} MB")

# Architecture
print("\n3. Architecture:")
print(f"   Embedding Dim: {MODEL_CONFIG['d_model']}")
print(f"   Num Layers: {MODEL_CONFIG['num_layers']}")
print(f"   Num Heads: {MODEL_CONFIG['num_heads']}")
print(f"   Vocabulary Size: {MODEL_CONFIG['vocab_size']}")

print("\n" + "="*60)